# Automatic generation of Notebook using PyCropML
    This notebook implements a crop model.

### Model Referenceevapotranspiration

In [ ]:
MODULE list_sub
    IMPLICIT NONE
    TYPE container
        CLASS(*), ALLOCATABLE :: item
        CLASS(*), ALLOCATABLE :: items(:)
    END TYPE

    interface Add
        module procedure AddToListFloat
        module procedure AddToListInt
        module procedure AddToListChar
        module procedure AddToListIntArray
    end interface
CONTAINS

    FUNCTION indice(vectorElem, elem)
        CHARACTER(LEN=*), DIMENSION(:):: vectorElem
        INTEGER::iterator, indice
        CHARACTER(LEN=*):: elem
        DO iterator= 1, SIZE(vectorElem)
            IF(vectorElem(iterator)==elem) THEN
                indice = iterator
            END IF
        END DO
        RETURN
    END FUNCTION indice


    SUBROUTINE AddToListFloat(list, element)
        IMPLICIT NONE
        INTEGER :: i, isize
        REAL, INTENT(IN) :: element
        REAL, DIMENSION(:), ALLOCATABLE, INTENT(INOUT) :: list
        REAL, DIMENSION(:), ALLOCATABLE :: clist
        IF(ALLOCATED(list)) THEN
            isize = SIZE(list)
            ALLOCATE(clist(isize+1))
            DO i=1,isize
                clist(i) = list(i)
            END DO
            clist(isize+1) = element
            DEALLOCATE(list)
            CALL MOVE_ALLOC(clist, list)
        ELSE
            ALLOCATE(list(1))
            list(1) = element
        END IF
    END SUBROUTINE AddToListFloat

    SUBROUTINE AddToListInt(list, element)
        IMPLICIT NONE
        INTEGER :: i, isize
        INTEGER, INTENT(IN) :: element
        INTEGER, DIMENSION(:), ALLOCATABLE, INTENT(INOUT) :: list
        INTEGER, DIMENSION(:), ALLOCATABLE :: clist
        IF(ALLOCATED(list)) THEN
            isize = SIZE(list)
            ALLOCATE(clist(isize+1))
            DO i=1,isize
                clist(i) = list(i)
            END DO
            clist(isize+1) = element
            DEALLOCATE(list)
            CALL MOVE_ALLOC(clist, list)
        ELSE
            ALLOCATE(list(1))
            list(1) = element
        END IF
    END SUBROUTINE AddToListInt

    SUBROUTINE AddToListChar(list, element)
        IMPLICIT NONE
        INTEGER :: i, isize, l
        CHARACTER(LEN=*), INTENT(IN) :: element
        CHARACTER(LEN=*), DIMENSION(:), ALLOCATABLE, INTENT(INOUT) :: list
        CHARACTER(LEN=65), DIMENSION(:), ALLOCATABLE :: clist
        IF(ALLOCATED(list)) THEN
            isize = SIZE(list)
            ALLOCATE(clist(isize+1))
            DO i=1,isize
                clist(i) = list(i)
            END DO
            clist(isize+1) = element
            DEALLOCATE(list)
            CALL MOVE_ALLOC(clist, list)
        ELSE
            l=1
            ALLOCATE(list(l))
            list(l) = element
        END IF
    END SUBROUTINE AddToListChar


    SUBROUTINE AddToListIntArray(a, e)
        INTEGER, ALLOCATABLE, INTENT(INOUT) :: a(:)
        INTEGER, ALLOCATABLE, INTENT(IN) :: e(:)
        INTEGER, ALLOCATABLE:: tmp(:)

        IF (.NOT.ALLOCATED(a)) THEN
            ALLOCATE(a(1))
            ALLOCATE(a(SIZE(e)), source = e)
        ELSE
            CALL MOVE_ALLOC(a,tmp)
            ALLOCATE(a(SIZE(tmp)+SIZE(e)), source = e)
            a(1:SIZE(tmp)) = tmp
            a(SIZE(tmp)+1:SIZE(a)) = e
        END IF
    END SUBROUTINE AddToListIntArray

END MODULE list_sub
MODULE Referenceevapotranspirationmod
    IMPLICIT NONE
CONTAINS

    SUBROUTINE model_referenceevapotranspiration(height_nn, &
        mean_air_temperature, &
        wind_speed, &
        wind_speed_height, &
        net_radiation, &
        stomata_resistance, &
        saturation_vapor_pressure_deficit, &
        reference_evapotranspiration)
        IMPLICIT NONE
        INTEGER:: i_cyml_r
        REAL, INTENT(IN) :: height_nn
        REAL, INTENT(IN) :: mean_air_temperature
        REAL, INTENT(IN) :: wind_speed
        REAL, INTENT(IN) :: wind_speed_height
        REAL, INTENT(IN) :: net_radiation
        REAL, INTENT(IN) :: stomata_resistance
        REAL, INTENT(IN) :: saturation_vapor_pressure_deficit
        REAL, INTENT(OUT) :: reference_evapotranspiration
        REAL:: atmospheric_pressure
        REAL:: psycrometer_constant
        REAL:: saturated_vapour_pressure_slope
        REAL:: wind_speed_2m
        REAL:: aerodynamic_resistance
        REAL:: surface_resistance
        !- Name: ReferenceEvapotranspiration -Version: 1, -Time step: 1
        !- Description:
    !            * Title: MONICA reference evapotranspiration
    !            * Authors: Claas Nendel (transcription into Crop2ML by Michael Berg-Mohnicke)
    !            * Reference: 
    !            Allen RG, Pereira LS, Raes D, Smith M. (1998) Crop evapotranspiration. Guidelines for computing crop water requirements. FAO Irrigation and
    !            Drainage Paper 56, FAO, Roma
    !        
    !            * Institution: ZALF e.V.
    !            * ExtendedDescription: 
    !            A method following Penman-Monteith as described by the FAO in Allen
    !            RG, Pereira LS, Raes D, Smith M. (1998) Crop evapotranspiration.
    !            Guidelines for computing crop water requirements. FAO Irrigation and
    !            Drainage Paper 56, FAO, Roma
    !        
    !            * ShortDescription: Calculates the MONICA reference evapotranspiration
        !- inputs:
    !            * name: height_nn
    !                          ** description : height above sea level
    !                          ** inputtype : parameter
    !                          ** parametercategory : constant
    !                          ** datatype : DOUBLE
    !                          ** max : 9999
    !                          ** min : -9999
    !                          ** default : 0
    !                          ** unit : m
    !            * name: mean_air_temperature
    !                          ** description : daily average air temperature
    !                          ** inputtype : variable
    !                          ** variablecategory : exogenous
    !                          ** datatype : DOUBLE
    !                          ** max : 100
    !                          ** min : -100
    !                          ** default : 0
    !                          ** unit : °C
    !            * name: wind_speed
    !                          ** description : wind speed measured at wind speed height
    !                          ** inputtype : variable
    !                          ** variablecategory : exogenous
    !                          ** datatype : DOUBLE
    !                          ** max : 9999
    !                          ** min : 0
    !                          ** default : 0
    !                          ** unit : m/s
    !            * name: wind_speed_height
    !                          ** description : height at which the wind speed has been measured
    !                          ** inputtype : variable
    !                          ** variablecategory : exogenous
    !                          ** datatype : DOUBLE
    !                          ** max : 9999
    !                          ** min : -9999
    !                          ** default : 2
    !                          ** unit : m
    !            * name: net_radiation
    !                          ** description : net radiation
    !                          ** inputtype : variable
    !                          ** variablecategory : auxiliary
    !                          ** datatype : DOUBLE
    !                          ** max : 
    !                          ** min : 0
    !                          ** unit : MJ/m2
    !            * name: stomata_resistance
    !                          ** description : stomata resistance (FAO default value is 100)
    !                          ** inputtype : variable
    !                          ** variablecategory : auxiliary
    !                          ** datatype : DOUBLE
    !                          ** max : 10000
    !                          ** min : 0
    !                          ** default : 100
    !                          ** unit : s/m
    !            * name: saturation_vapor_pressure_deficit
    !                          ** description : saturation_vapor_pressure_deficit
    !                          ** inputtype : variable
    !                          ** variablecategory : auxiliary
    !                          ** datatype : DOUBLE
    !                          ** max : 
    !                          ** min : 0
    !                          ** default : 0
    !                          ** unit : kPa
        !- outputs:
    !            * name: reference_evapotranspiration
    !                          ** description : reference evapotranspiration (ET0)
    !                          ** variablecategory : auxiliary
    !                          ** datatype : DOUBLE
    !                          ** max : 
    !                          ** min : 0
    !                          ** unit : mm
        atmospheric_pressure = 101.3 *  (((293.0 - (0.0065 * height_nn)) /  &
                293.0) ** 5.26)
        psycrometer_constant = 0.000665 * atmospheric_pressure
        saturated_vapour_pressure_slope = 4098.0 * (0.6108 * EXP(17.27 *  &
                mean_air_temperature / (mean_air_temperature + 237.3))) /  &
                ((mean_air_temperature + 237.3) * (mean_air_temperature + 237.3))
        wind_speed_2m = MAX(0.5, wind_speed * (4.87 / LOG((67.8 *  &
                wind_speed_height - 5.42))))
        aerodynamic_resistance = 208.0 / wind_speed_2m
        surface_resistance = stomata_resistance / 1.44
        reference_evapotranspiration = (0.408 *  &
                saturated_vapour_pressure_slope * net_radiation +  &
                (psycrometer_constant * (900.0 / (mean_air_temperature + 273.0)) *  &
                wind_speed_2m * saturation_vapor_pressure_deficit)) /  &
                (saturated_vapour_pressure_slope + (psycrometer_constant * (1.0 +  &
                (surface_resistance / aerodynamic_resistance))))
        IF(reference_evapotranspiration .LT. 0.0) THEN
            reference_evapotranspiration = 0.0
        END IF
    END SUBROUTINE model_referenceevapotranspiration

END MODULE
PROGRAM test
    USE Referenceevapotranspirationmod

END PROGRAM
